In [1]:
import glob
import os
import re
import sys

import numpy as np
import optuna
import pandas as pd

/panfs/jay/groups/33/kuangr/inoue019/conda-envs/myenv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from drGAT.metrics import get_parsed_df

In [3]:
def extract_method_data_cell_or_drug(filename):
    # ファイル名から method, data, cell_or_drug を抽出する関数を想定（例: method_data_cell.sqlite3）
    parts = filename.replace(".sqlite3", "").split("_")
    method = parts[0]
    data = parts[1]
    cell_or_drug = "_".join(parts[2:])  # 複数アンダースコアを許容
    return method, data, cell_or_drug

['no_atten_ctrp_cell.sqlite3',
 'no_atten_ctrp_drug.sqlite3',
 'no_atten_gdsc1_cell.sqlite3',
 'no_atten_gdsc1_drug.sqlite3',
 'no_atten_gdsc2_cell.sqlite3',
 'no_atten_gdsc2_drug.sqlite3',
 'no_atten_nci_cell.sqlite3',
 'no_atten_nci_drug.sqlite3']

In [11]:
tmp = !ls Test2_leave_X_out/ | grep sqlite | grep no_atten
tmp

['no_atten_ctrp_cell.sqlite3',
 'no_atten_ctrp_drug.sqlite3',
 'no_atten_gdsc1_cell.sqlite3',
 'no_atten_gdsc1_drug.sqlite3',
 'no_atten_gdsc2_cell.sqlite3',
 'no_atten_gdsc2_drug.sqlite3',
 'no_atten_nci_cell.sqlite3',
 'no_atten_nci_drug.sqlite3']

In [14]:
method, data, cell_or_drug

('no', 'atten', 'nci_drug')

'ctrp'

In [33]:
all_dfs = []

for file in tmp:
    data, cell_or_drug = file.split('_')[2], file.split('_')[3].split('.')[0]
    try:
        study = optuna.load_study(study_name='MPNN_GCN', storage=f"sqlite:///Test2_leave_X_out/{file}")
        df_all = study.trials_dataframe()

        n_complete = (df_all["state"] == "COMPLETE").sum()
        print(f"✅ {file}: {n_complete} trials completed")

        df_valid = df_all.dropna(
            subset=["values_0", "values_1", "values_2", "values_3"]
        )

        for gnn_type in ['MPNN', 'GCN']:
            df_method = df_valid[df_valid.params_gnn_layer == gnn_type]

            if df_method.shape[0] == 0:
                # 試行ゼロのときのダミー行
                empty_row = pd.DataFrame([{
                    "n_complete": 0,
                    "method": gnn_type,
                    "data": data,
                    "cell_or_drug": cell_or_drug,
                }])
                all_dfs.append(empty_row)
                continue

            df_metrics = df_method.iloc[:, 20:-2]
            df_metrics.columns = [c.replace("user_attrs_", "") for c in df_metrics.columns]
            parsed_df = get_parsed_df(df_metrics)
            parsed_df["n_complete"] = n_complete

            df_params = df_method[[c for c in df_method.columns if "params" in c]].reset_index(drop=True)
            parsed_df = pd.concat([parsed_df.reset_index(drop=True), df_params], axis=1)

            parsed_df["method"] = gnn_type
            parsed_df["data"] = data
            parsed_df["cell_or_drug"] = cell_or_drug

            all_dfs.append(parsed_df)

    except Exception as e:
        print(f"❌ Failed to parse {file}: {e}")

if all_dfs:
    summary_df = pd.concat(all_dfs, ignore_index=True)
    summary_df["AUPR_num"] = summary_df["AUPR"].str.extract(r"([\d\.]+)").astype(float)

    # 試行ありの部分だけで idxmax をとる
    valid_summary_df = summary_df[summary_df["n_complete"] > 0].copy()
    best_idxs = valid_summary_df.groupby(["data", "cell_or_drug", "method"])["AUPR_num"].idxmax()
    best_valid_df = summary_df.loc[best_idxs]

    # 試行ゼロの行だけ抽出
    zero_trial_df = summary_df[summary_df["n_complete"] == 0]

    # 両者を結合
    best_df = pd.concat([best_valid_df, zero_trial_df], ignore_index=True).drop(columns=["AUPR_num"])

    # インデックスと並び順
    best_df.set_index(["data", "cell_or_drug", "method"], inplace=True)
    best_df.sort_index(level=["data", "cell_or_drug"], inplace=True)

    desired_order = ["n_complete", "ACC", "Precision", "Recall", "F1", "AUROC", "AUPR"]
    other_cols = [col for col in best_df.columns if col not in desired_order]
    best_df = best_df[desired_order + other_cols]

    desired_data_order = ["nci", "gdsc1", "gdsc2", "ctrp"]
    best_df = best_df.reindex(
        best_df.index.reindex(
            sorted(best_df.index, key=lambda x: (desired_data_order.index(x[0]), x[1]))
        )[0]
    )

    display(best_df)
else:
    print("No valid studies found.")


✅ no_atten_ctrp_cell.sqlite3: 7 trials completed
✅ no_atten_ctrp_drug.sqlite3: 2 trials completed
✅ no_atten_gdsc1_cell.sqlite3: 10 trials completed
✅ no_atten_gdsc1_drug.sqlite3: 3 trials completed
✅ no_atten_gdsc2_cell.sqlite3: 27 trials completed
✅ no_atten_gdsc2_drug.sqlite3: 8 trials completed
✅ no_atten_nci_cell.sqlite3: 8 trials completed
✅ no_atten_nci_drug.sqlite3: 49 trials completed


n_complete              ACC        Precision  \
data  cell_or_drug method                                                 
nci   cell         GCN              8  0.842 (± 0.139)  0.883 (± 0.153)   
                   MPNN             8  0.806 (± 0.146)  0.875 (± 0.174)   
      drug         GCN             49  0.503 (± 0.347)      0.0 (± 0.0)   
                   MPNN            49  0.504 (± 0.346)  0.029 (± 0.158)   
gdsc1 cell         GCN             10  0.677 (± 0.117)  0.625 (± 0.227)   
                   MPNN            10  0.632 (± 0.148)   0.57 (± 0.253)   
      drug         GCN              3  0.589 (± 0.232)  0.234 (± 0.333)   
                   MPNN             0              NaN              NaN   
gdsc2 cell         GCN             27  0.737 (± 0.138)  0.695 (± 0.244)   
                   MPNN            27  0.704 (± 0.166)    0.67 (± 0.27)   
      drug         GCN              8  0.618 (± 0.286)  0.157 (± 0.326)   
                   MPNN             8  0.695 (± 0.241)  0.305 (± 0.387)   
ctrp  cell         GCN              7  0.783 (± 0.128)  0.775 (± 0.178)   
                   MPNN             0              NaN              NaN   
      drug         GCN              2  0.551 (± 0.315)  0.046 (± 0.202)   
                   MPNN             0              NaN              NaN   

                                    Recall               F1            AUROC  \
data  cell_or_drug method                                                      
nci   cell         GCN       0.796 (± 0.2)  0.822 (± 0.165)  0.906 (± 0.124)   
                   MPNN    0.726 (± 0.228)  0.771 (± 0.189)  0.885 (± 0.137)   
      drug         GCN         0.0 (± 0.0)      0.0 (± 0.0)  0.517 (± 0.061)   
                   MPNN    0.001 (± 0.007)  0.002 (± 0.013)   0.51 (± 0.065)   
gdsc1 cell         GCN     0.622 (± 0.209)  0.593 (± 0.184)  0.738 (± 0.144)   
                   MPNN    0.541 (± 0.254)  0.522 (± 0.223)  0.683 (± 0.183)   
      drug         GCN     0.304 (± 0.423)  0.225 (± 0.324)  0.506 (± 0.145)   
                   MPNN                NaN              NaN              NaN   
gdsc2 cell         GCN     0.685 (± 0.266)  0.651 (± 0.231)  0.812 (± 0.147)   
                   MPNN    0.651 (± 0.295)  0.611 (± 0.257)  0.781 (± 0.161)   
      drug         GCN      0.168 (± 0.36)  0.149 (± 0.318)   0.52 (± 0.221)   
                   MPNN    0.421 (± 0.487)  0.335 (± 0.412)  0.501 (± 0.212)   
ctrp  cell         GCN     0.785 (± 0.184)  0.765 (± 0.163)   0.86 (± 0.129)   
                   MPNN                NaN              NaN              NaN   
      drug         GCN       0.01 (± 0.08)   0.01 (± 0.066)  0.512 (± 0.128)   
                   MPNN                NaN              NaN              NaN   

                                      AUPR     Balanced_ACC  params_T_max  \
data  cell_or_drug method                                                   
nci   cell         GCN     0.914 (± 0.118)  0.846 (± 0.133)           NaN   
                   MPNN    0.901 (± 0.124)   0.81 (± 0.141)           NaN   
      drug         GCN      0.52 (± 0.346)      0.5 (± 0.0)           NaN   
                   MPNN    0.519 (± 0.342)    0.5 (± 0.004)           NaN   
gdsc1 cell         GCN      0.695 (± 0.21)  0.679 (± 0.126)          23.0   
                   MPNN     0.66 (± 0.217)  0.627 (± 0.155)          18.0   
      drug         GCN     0.504 (± 0.249)  0.499 (± 0.046)           2.0   
                   MPNN                NaN              NaN           NaN   
gdsc2 cell         GCN     0.797 (± 0.188)  0.742 (± 0.145)          31.0   
                   MPNN    0.762 (± 0.203)  0.707 (± 0.151)           8.0   
      drug         GCN     0.574 (± 0.292)  0.499 (± 0.035)          32.0   
                   MPNN    0.562 (± 0.296)    0.5 (± 0.039)           NaN   
ctrp  cell         GCN     0.863 (± 0.146)  0.789 (± 0.126)           NaN   
                   MPNN                NaN              NaN           NaN   
   

In [34]:
output_df = best_df[desired_order].reset_index()
output_df

,data,cell_or_drug,method,n_complete,ACC,Precision,Recall,F1,AUROC,AUPR
0,nci,cell,GCN,8,0.842 (± 0.139),0.883 (± 0.153),0.796 (± 0.2),0.822 (± 0.165),0.906 (± 0.124),0.914 (± 0.118)
1,nci,cell,MPNN,8,0.806 (± 0.146),0.875 (± 0.174),0.726 (± 0.228),0.771 (± 0.189),0.885 (± 0.137),0.901 (± 0.124)
2,nci,drug,GCN,49,0.503 (± 0.347),0.0 (± 0.0),0.0 (± 0.0),0.0 (± 0.0),0.517 (± 0.061),0.52 (± 0.346)
3,nci,drug,MPNN,49,0.504 (± 0.346),0.029 (± 0.158),0.001 (± 0.007),0.002 (± 0.013),0.51 (± 0.065),0.519 (± 0.342)
4,gdsc1,cell,GCN,10,0.677 (± 0.117),0.625 (± 0.227),0.622 (± 0.209),0.593 (± 0.184),0.738 (± 0.144),0.695 (± 0.21)
5,gdsc1,cell,MPNN,10,0.632 (± 0.148),0.57 (± 0.253),0.541 (± 0.254),0.522 (± 0.223),0.683 (± 0.183),0.66 (± 0.217)
6,gdsc1,drug,GCN,3,0.589 (± 0.232),0.234 (± 0.333),0.304 (± 0.423),0.225 (± 0.324),0.506 (± 0.145),0.504 (± 0.249)
7,gdsc1,drug,MPNN,0,NaN,NaN,NaN,NaN,NaN,NaN
8,gdsc2,cell,GCN,27,0.737 (± 0.138),0.695 (± 0.244),0.685 (± 0.266),0.651 (± 0.231),0.812 (± 0.147),0.797 (± 0.188)
9,gdsc2,cell,MPNN,27,0.704 (± 0.166),0.67 (± 0.27),0.651 (± 0.295),0.611 (± 0.257),0.781 (± 0.161),0.762 (± 0.203)


In [35]:
output_df.sort_values('cell_or_drug')

,data,cell_or_drug,method,n_complete,ACC,Precision,Recall,F1,AUROC,AUPR
0,nci,cell,GCN,8,0.842 (± 0.139),0.883 (± 0.153),0.796 (± 0.2),0.822 (± 0.165),0.906 (± 0.124),0.914 (± 0.118)
1,nci,cell,MPNN,8,0.806 (± 0.146),0.875 (± 0.174),0.726 (± 0.228),0.771 (± 0.189),0.885 (± 0.137),0.901 (± 0.124)
4,gdsc1,cell,GCN,10,0.677 (± 0.117),0.625 (± 0.227),0.622 (± 0.209),0.593 (± 0.184),0.738 (± 0.144),0.695 (± 0.21)
5,gdsc1,cell,MPNN,10,0.632 (± 0.148),0.57 (± 0.253),0.541 (± 0.254),0.522 (± 0.223),0.683 (± 0.183),0.66 (± 0.217)
8,gdsc2,cell,GCN,27,0.737 (± 0.138),0.695 (± 0.244),0.685 (± 0.266),0.651 (± 0.231),0.812 (± 0.147),0.797 (± 0.188)
9,gdsc2,cell,MPNN,27,0.704 (± 0.166),0.67 (± 0.27),0.651 (± 0.295),0.611 (± 0.257),0.781 (± 0.161),0.762 (± 0.203)
12,ctrp,cell,GCN,7,0.783 (± 0.128),0.775 (± 0.178),0.785 (± 0.184),0.765 (± 0.163),0.86 (± 0.129),0.863 (± 0.146)
13,ctrp,cell,MPNN,0,NaN,NaN,NaN,NaN,NaN,NaN
2,nci,drug,GCN,49,0.503 (± 0.347),0.0 (± 0.0),0.0 (± 0.0),0.0 (± 0.0),0.517 (± 0.061),0.52 (± 0.346)
3,nci,drug,MPNN,49,0.504 (± 0.346),0.029 (± 0.158),0.001 (± 0.007),0.002 (± 0.013),0.51 (± 0.065),0.519 (± 0.342)


,number,values_0,values_1,values_2,values_3,datetime_start,datetime_complete,duration,params_T_max,params_activation,...,user_attrs_MCC_mean,user_attrs_MCC_std,user_attrs_Precision_mean,user_attrs_Precision_std,user_attrs_Recall_mean,user_attrs_Recall_std,user_attrs_Specificity_mean,user_attrs_Specificity_std,system_attrs_NSGAIISampler:generation,state
0,0,0.509371,0.511389,0.000119,0.503073,2025-05-31 20:27:26.407696,2025-05-31 20:56:35.739967,0 days 00:29:09.332271,NaN,gelu,...,0.000275,0.002077,0.017544,0.132453,0.000059,0.000449,1.000000,0.000000,0,COMPLETE
1,1,0.494960,0.506511,0.031040,0.503586,2025-05-31 20:31:55.746648,2025-05-31 20:33:05.121428,0 days 00:01:09.374780,7.0,relu,...,-0.003705,0.036421,0.088974,0.258060,0.036922,0.160445,0.962977,0.149185,0,COMPLETE
4,4,0.513683,0.513177,0.000000,0.503018,2025-05-31 20:40:48.871871,2025-05-31 20:42:40.810223,0 days 00:01:51.938352,NaN,relu,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0,COMPLETE
9,9,0.499463,0.506999,0.004812,0.505128,2025-05-31 20:48:41.140809,2025-05-31 20:49:59.270572,0 days 00:01:18.129763,NaN,gelu,...,0.000189,0.012802,0.050785,0.217680,0.002654,0.015772,0.996850,0.023034,0,COMPLETE
10,10,0.499207,0.510266,0.000000,0.503018,2025-05-31 20:48:51.040811,2025-05-31 20:51:02.293587,0 days 00:02:11.252776,NaN,gelu,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0,COMPLETE
13,13,0.512367,0.509459,0.000000,0.503018,2025-05-31 20:50:05.068858,2025-05-31 20:51:39.531641,0 days 00:01:34.462783,12.0,gelu,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0,COMPLETE
14,14,0.509312,0.515228,0.000000,0.502956,2025-05-31 20:51:02.434249,2025-05-31 21:18:45.759125,0 days 00:27:43.324876,34.0,gelu,...,-0.000358,0.002706,0.000000,0.000000,0.000000,0.000000,0.999931,0.000521,0,COMPLETE
15,15,0.519309,0.516911,0.028464,0.518529,2025-05-31 20:51:39.875574,2025-05-31 21:29:06.522216,0 days 00:37:26.646642,NaN,gelu,...,0.001562,0.010485,0.025827,0.142024,0.033260,0.175979,0.967576,0.171543,0,COMPLETE
19,19,0.494494,0.508316,0.033957,0.525739,2025-05-31 21:01:46.459329,2025-05-31 21:14:32.188448,0 days 00:12:45.729119,NaN,relu,...,0.000314,0.013576,0.041770,0.179262,0.034955,0.176803,0.965332,0.176941,0,COMPLETE
22,22,0.499091,0.502586,0.018892,0.507079,2025-05-31 21:19:32.805365,2025-05-31 21:20:49.020562,0 days 00:01:16.215197,23.0,gelu,...,-0.006836,0.051084,0.148951,0.336830,0.011078,0.038465,0.986307,0.040630,0,COMPLETE


,number,values_0,values_1,values_2,values_3,datetime_start,datetime_complete,duration,params_T_max,params_activation,...,user_attrs_MCC_mean,user_attrs_MCC_std,user_attrs_Precision_mean,user_attrs_Precision_std,user_attrs_Recall_mean,user_attrs_Recall_std,user_attrs_Specificity_mean,user_attrs_Specificity_std,system_attrs_NSGAIISampler:generation,state
2,2,0.497876,0.507304,0.066296,0.464270,2025-05-31 20:31:57.544424,2025-05-31 20:40:48.613642,0 days 00:08:51.069218,NaN,gelu,...,0.003789,0.023228,0.090869,0.219573,0.127724,0.324408,0.874828,0.323282,0,COMPLETE
3,3,0.500514,0.509924,0.036921,0.512442,2025-05-31 20:33:05.239591,2025-05-31 20:48:40.990136,0 days 00:15:35.750545,NaN,gelu,...,-0.004785,0.027424,0.110243,0.274229,0.041331,0.185263,0.955931,0.185814,0,COMPLETE
5,5,0.514023,0.516294,0.000000,0.503018,2025-05-31 20:42:41.059413,2025-05-31 20:44:53.941078,0 days 00:02:12.881665,NaN,gelu,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0,COMPLETE
6,6,0.508027,0.513436,0.003899,0.495956,2025-05-31 20:44:54.532784,2025-05-31 20:47:58.950032,0 days 00:03:04.417248,9.0,gelu,...,0.000351,0.002649,0.002411,0.018200,0.010187,0.076908,0.990329,0.073011,0,COMPLETE
7,7,0.518397,0.519462,0.000000,0.503018,2025-05-31 20:47:22.901993,2025-05-31 20:48:50.780408,0 days 00:01:27.878415,27.0,gelu,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0,COMPLETE
8,8,0.509806,0.509867,0.000000,0.503018,2025-05-31 20:47:59.396201,2025-05-31 20:49:11.059188,0 days 00:01:11.662987,NaN,relu,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0,COMPLETE
11,11,0.504463,0.505171,0.000595,0.503269,2025-05-31 20:49:11.242791,2025-05-31 20:50:04.798240,0 days 00:00:53.555449,16.0,gelu,...,0.000961,0.007253,0.017544,0.132453,0.000302,0.002284,1.000000,0.000000,0,COMPLETE
12,12,0.480622,0.499310,0.018248,0.518217,2025-05-31 20:49:59.507405,2025-05-31 20:57:48.402015,0 days 00:07:48.894610,NaN,relu,...,-0.001209,0.009130,0.026809,0.145185,0.018270,0.132469,0.981132,0.132652,0,COMPLETE
17,17,0.514096,0.517167,0.000000,0.503018,2025-05-31 20:57:48.571832,2025-05-31 20:59:18.146668,0 days 00:01:29.574836,NaN,gelu,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0,COMPLETE
18,18,0.515519,0.517452,0.089424,0.507346,2025-05-31 20:59:18.265858,2025-05-31 21:01:46.337502,0 days 00:02:28.071644,NaN,gelu,...,-0.001881,0.028218,0.132441,0.310738,0.142106,0.336756,0.856498,0.335125,0,COMPLETE
